# 🧠 Amazon Fine Review - Análisis de Sentimientos y NLP
Este notebook documenta el pipeline de Procesamiento de Lenguaje Natural (NLP) aplicado a las reseñas de Amazon. Está diseñado para mostrar la limpieza de datos, extracción de sentimientos mediante VADER y TextBlob, e identificación de temas principales asociados a reseñas negativas.

## 1. Configuración y Carga de Datos
Importamos librerías y cargamos una muestra representativa de los datos de reseñas.

In [ ]:
import pandas as pd
import numpy as np
import spacy
import re
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load a sample from Amazon Fine Food Reviews dataset
try:
    df = pd.read_csv('data/Reviews.csv').sample(10000, random_state=42).copy()
except FileNotFoundError:
    print("Dataset not found locally. Skipping exact load.")
    df = pd.DataFrame({'Score': [5, 1], 'Summary': ['Great!', 'Bad'], 'Text': ['Good product.', 'Terrible.']})
df['Text'] = df['Text'].fillna('')
print("Sample Shape:", df.shape)
df[['Score', 'Summary', 'Text']].head(3)

## 2. Limpieza y Preprocesamiento de Texto
Aplicamos expresiones regulares y lematización (usando SpaCy) para preparar el texto para el análisis de sentimientos, eliminando ruido como URLs y etiquetas HTML.

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def clean_text(text):
    t = text.lower()
    t = re.sub(r'<[^>]+>', ' ', t)
    t = re.sub(r'http\S+', ' ', t)
    t = re.sub(r'[^a-zA-Z\s]', ' ', t)
    return t

df['PreClean'] = df['Text'].apply(clean_text)

cleaned_texts = []
for doc in nlp.pipe(df['PreClean'].tolist(), batch_size=500):
    tokens = [token.lemma_ for token in doc if len(token.text) > 1]
    cleaned_texts.append(" ".join(tokens))
    
df['CleanText'] = cleaned_texts
df[['PreClean', 'CleanText']].head(3)

## 3. Análisis de Sentimientos (VADER & TextBlob)
Calculamos un puntaje compuesto de sentimiento para cada reseña, promediando los resultados de VADER y TextBlob para mayor precisión.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

df['Vader_Score'] = df['Text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
df['TextBlob_Score'] = df['Text'].apply(lambda x: TextBlob(x).sentiment.polarity)

df['Sentiment_Score'] = (df['Vader_Score'] + df['TextBlob_Score']) / 2

def classify_sentiment(score):
    if score > 0.05: return "Positive"
    elif score < -0.05: return "Negative"
    else: return "Neutral"
    
df['Sentiment_Class'] = df['Sentiment_Score'].apply(classify_sentiment)
print(df['Sentiment_Class'].value_counts(normalize=True)*100)

## 4. Modelado de Temas en Reseñas Negativas
Para las reseñas clasificadas como negativas, extraemos los temas predominantes basados en palabras clave predefinidas (ej. 'calidad', 'entrega', 'precio').

In [ ]:
neg_reviews = df[df['Sentiment_Class'] == 'Negative'].copy()

categories = {
    "entrega/envío": ["deliver", "ship", "arrive", "box", "package"],
    "calidad": ["taste", "stale", "bad", "horrible", "awful"],
    "precio": ["price", "money", "expensive", "cost", "waste"]
}

def assign_category(text):
    text_lower = text.lower()
    counts = {cat: sum([1 for word in words if word in text_lower]) for cat, words in categories.items()}
    best_cat = max(counts, key=counts.get)
    if counts[best_cat] > 0:
        return best_cat
    return "otros"
    
if not neg_reviews.empty:
    neg_reviews['Dominant_Topic'] = neg_reviews['CleanText'].apply(assign_category)
    print("Negative Topics Frequency:\n", neg_reviews['Dominant_Topic'].value_counts())

## 5. Exportación de Datos
Finalmente, organizamos las columnas necesarias para el Dashboard Interactivo de Streamlit y guardamos el dataset final.

In [ ]:
# Simulate Category and Date for Dashboard integration
import random
product_categories = ["Snacks", "Beverages", "Pantry", "Candy", "Baby Food"]
df['Category'] = [random.choice(product_categories) for _ in range(len(df))]

if 'Time' in df.columns:
    df['Date'] = pd.to_datetime(df['Time'], unit='s')
else:
    df['Date'] = pd.to_datetime('today')

cols = ['Score', 'Date', 'Category', 'Sentiment_Score', 'Sentiment_Class']
cols = [c for c in cols if c in df.columns]
clean_df = df[cols]
print("Pipeline successful! Shape:", clean_df.shape)